In [ ]:
%%capture
!pip uninstall -y torch torchvision torchaudio triton bitsandbytes
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate bitsandbytes triton


### Data Preprocessing

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset


# Load a tokenizer to use its chat template
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", trust_remote_code=True)

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""

    # Format answers
    chat = [
    {"role": "user", "content": example['sentence']},
    {"role": "assistant", "content": example['corrections'][0]}
    ]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False)

    return {"text": prompt}

# Load and format the data using the template TinyLLama is using
dataset = (
    load_dataset("jhu-clsp/jfleg",  split="validation")
      .shuffle(seed=42)
)
dataset = dataset.map(format_prompt)

In [ ]:
print(dataset["text"][25])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Jan 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

I will explain my points of views in the following paragraphs .<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I will explain my point of view in the following paragraphs .<|eot_id|>


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",  # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # Leave this out for regular SFT
    quantization_config=bnb_config,
    trust_remote_code=True # Add this line
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True) # Change to True
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling
    lora_dropout=0.1,  # Dropout for LoRA Layers
    r=16,  # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

# Training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=100,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    save_steps=50,
    bf16=True,
    fp16=False,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    gradient_checkpointing=True

)

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 13.6 MB/s eta 0:00:00


In [ ]:
from trl import SFTTrainer

# Set up the trainer
model.enable_input_require_grads()

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,   # or SFTConfig
    formatting_func=formatting_func,
)

trainer.train()
trainer.model.save_pretrained("LiquidAI-grammarly")


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Step,Training Loss
10,2.975300
20,1.523200
30,1.533400
40,1.380300
50,1.258400
60,1.306400
70,1.244900
80,1.196300
90,1.306400
100,1.253700


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


### Merge Adapter


In [ ]:
trainer.model.save_pretrained("LiquidAI-grammarly")
tokenizer.save_pretrained("LiquidAI-grammarly")

('LiquidAI-grammarly/tokenizer_config.json',
 'LiquidAI-grammarly/special_tokens_map.json',
 'LiquidAI-grammarly/chat_template.jinja',
 'LiquidAI-grammarly/tokenizer.json')

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "LiquidAI-grammarly",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

### Inference

In [ ]:
from transformers import pipeline

# Use our predefined prompt template
sentence = """Write this sentence correctly: Here was no promise of morning except that we looked up through the trees we saw how low the forest had swung .
"""
dic1 = [{'role': "user", 'content': sentence}]

prompt = tokenizer.apply_chat_template(dic1, tokenize=False, add_generation_prompt=True)
# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

Device set to use cuda:0


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Jan 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Write this sentence correctly: Here was no promise of morning except that we looked up through the trees we saw how low the forest had swung .<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here was no promise of morning, except that we looked up through the trees, and we saw how low the forest had swung.


### Pushing it to hugging face

In [ ]:


trainer.model.push_to_hub(
    "arjunverma2004/LiquidAI-grammarly-lora",
    private=False,   # or True if you want
)
tokenizer.push_to_hub("arjunverma2004/LiquidAI-grammarly-lora")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 22.6MB / 22.6MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpjb6momam/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/arjunverma2004/LiquidAI-grammarly-lora/commit/59c4fd3bf129b6a80b00250b97678875ea9878e0', commit_message='Upload tokenizer', commit_description='', oid='59c4fd3bf129b6a80b00250b97678875ea9878e0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/arjunverma2004/LiquidAI-grammarly-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='arjunverma2004/LiquidAI-grammarly-lora'), pr_revision=None, pr_num=None)